# Feature engineering — internações respiratórias x qualidade do ar

Este notebook constrói o dataset final de modelagem para prever o número de internações por doenças respiratórias em `D0`, combinando informações da própria série de internações com variáveis atmosféricas e componentes sazonais.

A construção das features segue a lógica definida em `featureDefinition.md` e respeita o princípio de **anti-leakage**: nenhuma variável derivada da série de internações utiliza informação do próprio `D0` ou de datas futuras.

**Entradas**
- `respiratory_hospitalization_time_series.parquet`
- `serie_diaria_qualidade_ar_rio_de_janeiro.parquet`

**Saída**
- `Data/GoldData/modelDataset.parquet`


## Importações e configuração inicial

Nesta etapa são importadas as bibliotecas necessárias e definidos os parâmetros básicos de execução do notebook.


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

## Caminhos dos dados e carregamento das bases

Aqui são definidos os caminhos das duas bases utilizadas no processo e realizado o carregamento inicial dos dados em memória.


In [33]:
BASE_DIR = Path('../../Data')

PATH_INTERNACOES = 'https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/DataSus/respiratory_hospitalization_time_series.parquet'
PATH_QUALIAR = 'https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/DailyQualiarRj/serie_diaria_qualidade_ar_rio_de_janeiro.parquet'

PATH_OUTPUT = BASE_DIR / 'GoldData' / 'modelDataset.parquet'

# Carrega as bases de dados
df_internacoes = pd.read_parquet(PATH_INTERNACOES)
df_qualiar = pd.read_parquet(PATH_QUALIAR)

print(f'Serie de internacoes: {df_internacoes.shape}')
print(f'  Periodo: {df_internacoes["data_dia"].min()} a {df_internacoes["data_dia"].max()}')
print(f'\nSerie de qualidade do ar: {df_qualiar.shape}')
print(f'  Periodo: {df_qualiar["data"].min()} a {df_qualiar["data"].max()}')

Serie de internacoes: (2557, 2)
  Periodo: 2012-01-01 00:00:00 a 2018-12-31 00:00:00

Serie de qualidade do ar: (2557, 12)
  Periodo: 2012-01-01 00:00:00 a 2018-12-31 00:00:00


## Preparação das datas, merge e tratamento inicial

As colunas de data são padronizadas para o mesmo formato, as duas bases são integradas por dia e é feita uma checagem de continuidade temporal.

Também é aplicado um tratamento pontual de valores ausentes nas variáveis atmosféricas, com interpolação linear, para evitar propagação excessiva de `NaN` nas janelas móveis mais longas.


In [24]:
df_internacoes['data'] = pd.to_datetime(df_internacoes['data_dia']).dt.normalize()
df_qualiar['data'] = pd.to_datetime(df_qualiar['data']).dt.normalize()

df = pd.merge(
    df_internacoes[['data', 'num_internacoes']],
    df_qualiar,
    on='data',
    how='inner'
)
df = df.sort_values('data').reset_index(drop=True)

print(f'Dataset apos merge: {df.shape}')
print(f'Periodo: {df["data"].min().date()} a {df["data"].max().date()}')
print(f'Dias totais: {len(df)}')
print(f'Dias esperados no periodo: {(df["data"].max() - df["data"].min()).days + 1}')

# Verifica se ha lacunas na serie temporal
datas_esperadas = pd.date_range(df['data'].min(), df['data'].max(), freq='D')
datas_faltantes = datas_esperadas.difference(df['data'])
print(f'Dias faltantes na serie: {len(datas_faltantes)}')
if len(datas_faltantes) > 0:
    print(f'  Primeiros faltantes: {sorted(datas_faltantes)[:5]}')

# Tratamento de NaN nas variaveis atmosfericas
# As estacoes de monitoramento apresentam poucas falhas pontuais (< 1% dos dias).
# Interpolacao linear e usada para preencher essas lacunas e evitar propagacao
# de NaN nas rolling windows longas (120-150 dias), que amplificariam a perda
# de dados de forma desproporcional. Isso nao configura data leakage pois:
#   (a) sao variaveis exogenas, nao o alvo;
#   (b) a interpolacao preenche apenas lacunas minimas em sensores.
vars_atmosfericas = ['no', 'no2', 'so2', 'pm2_5', 'nox', 'ur', 'temp', 'o3', 'co', 'pm10']
n_nulos_antes = df[vars_atmosfericas].isnull().sum()
df[vars_atmosfericas] = df[vars_atmosfericas].interpolate(method='linear', limit_direction='both')
n_nulos_depois = df[vars_atmosfericas].isnull().sum()

print(f'\nNaN em variaveis atmosfericas (antes -> depois da interpolacao):')
for var in vars_atmosfericas:
    print(f'  {var}: {n_nulos_antes[var]} -> {n_nulos_depois[var]}')

Dataset apos merge: (2557, 13)
Periodo: 2012-01-01 a 2018-12-31
Dias totais: 2557
Dias esperados no periodo: 2557
Dias faltantes na serie: 0

NaN em variaveis atmosfericas (antes -> depois da interpolacao):
  no: 23 -> 0
  no2: 23 -> 0
  so2: 23 -> 0
  pm2_5: 41 -> 0
  nox: 23 -> 0
  ur: 23 -> 0
  temp: 23 -> 0
  o3: 23 -> 0
  co: 23 -> 0
  pm10: 23 -> 0


## Definição da variável alvo

O alvo do modelo é o número de internações no dia corrente (`D0`). A partir daqui, todas as features são construídas para prever esse valor utilizando apenas informações observáveis até o instante da previsão.


In [25]:
df['target'] = df['num_internacoes'].copy()

print('Alvo (target) — estatisticas descritivas:')
print(df['target'].describe())

Alvo (target) — estatisticas descritivas:
count    2557.000000
mean      370.944466
std       185.703771
min        48.000000
25%       234.000000
50%       340.000000
75%       468.000000
max      1218.000000
Name: target, dtype: float64


## Features endógenas — lags da série de internações

Nesta seção são criadas variáveis de memória temporal da própria série de internações. Cada `lag_k` representa o valor observado `k` dias antes de `D0`.

Essas features ajudam o modelo a capturar persistência de curto prazo, recorrência semanal e padrões de escala mais longa.


In [26]:
for k in [1, 2, 3]:
    df[f'lag_{k}'] = df['num_internacoes'].shift(k)

# Memoria semanal: recorrencia semanal foi um dos sinais mais fortes na EDA
for k in [7, 14, 21, 28]:
    df[f'lag_{k}'] = df['num_internacoes'].shift(k)

# Memoria mensal e anual: capturam ciclos de escala mais longa
for k in [30, 365]:
    df[f'lag_{k}'] = df['num_internacoes'].shift(k)

# Resumo dos lags criados
lag_cols = [c for c in df.columns if c.startswith('lag_')]
print(f'Lags criados ({len(lag_cols)}): {lag_cols}')
print(f'\nNaN por feature (linhas sem historico suficiente):')
for c in lag_cols:
    print(f'  {c}: {df[c].isnull().sum()} NaNs')

Lags criados (9): ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_30', 'lag_365']

NaN por feature (linhas sem historico suficiente):
  lag_1: 1 NaNs
  lag_2: 2 NaNs
  lag_3: 3 NaNs
  lag_7: 7 NaNs
  lag_14: 14 NaNs
  lag_21: 21 NaNs
  lag_28: 28 NaNs
  lag_30: 30 NaNs
  lag_365: 365 NaNs


## Features endógenas — janelas móveis da série de internações

Aqui são criadas estatísticas móveis da série de internações, sempre calculadas sobre a série deslocada em `1` dia, para impedir que o valor de `D0` entre na construção da feature.

São incluídas medidas de nível recente e de variabilidade recente da série.


In [27]:
serie_shifted = df['num_internacoes'].shift(1)

# Nivel recente: resumem o patamar local da serie e ajudam a capturar
# ondas epidemiologicas de diferentes escalas
for w in [7, 14, 30]:
    df[f'rolling_mean_{w}'] = serie_shifted.rolling(w).mean()

# Variabilidade recente: capturam mudancas de volatilidade, importantes
# numa serie com quebras estruturais e extremos
for w in [7, 30]:
    df[f'rolling_std_{w}'] = serie_shifted.rolling(w).std()

# Resumo das rolling windows criadas
roll_cols = [c for c in df.columns if c.startswith('rolling_')]
print(f'Rolling windows criadas ({len(roll_cols)}):')
for c in roll_cols:
    print(f'  {c}: {df[c].isnull().sum()} NaNs')

Rolling windows criadas (5):
  rolling_mean_7: 7 NaNs
  rolling_mean_14: 14 NaNs
  rolling_mean_30: 30 NaNs
  rolling_std_7: 7 NaNs
  rolling_std_30: 30 NaNs


## Features de calendário e sazonalidade

Esta etapa adiciona atributos determinísticos baseados na própria data, como dia da semana, mês, semana epidemiológica, estação do ano e indicador de período sazonal crítico.

Como essas informações são conhecidas antecipadamente, elas não oferecem risco de vazamento de informação.


In [28]:
df['dia_semana'] = df['data'].dt.dayofweek

# Fim de semana: a EDA mostrou efeito semanal forte, com dias uteis muito
# acima de sabado/domingo
df['fim_de_semana'] = (df['dia_semana'] >= 5).astype(int)

# Mes do ano (1-12)
df['mes'] = df['data'].dt.month

# Semana epidemiologica (aproximacao via ISO week)
# A semana epidemiologica brasileira segue convencao muito proxima da ISO,
# com diferenca menor no dia de inicio (domingo vs segunda)
df['semana_epidemiologica'] = df['data'].dt.isocalendar().week.astype(int)

# Estacao do ano — hemisferio sul (Rio de Janeiro)
# Verao: Dez-Fev (0) | Outono: Mar-Mai (1) | Inverno: Jun-Ago (2) | Primavera: Set-Nov (3)
ESTACAO_HEMISFERIO_SUL = {
    12: 0, 1: 0, 2: 0,    # Verao
    3: 1, 4: 1, 5: 1,     # Outono
    6: 2, 7: 2, 8: 2,     # Inverno
    9: 3, 10: 3, 11: 3    # Primavera
}
df['estacao'] = df['mes'].map(ESTACAO_HEMISFERIO_SUL)

# Periodo de pico sazonal: abril a julho (meses 4 a 7)
# Conforme a EDA, esse periodo concentra a maior carga sazonal de internacoes
df['dummy_periodo_pico_sazonal'] = df['mes'].isin([4, 5, 6, 7]).astype(int)

# Resumo das features de calendario
cal_cols = ['dia_semana', 'fim_de_semana', 'mes', 'semana_epidemiologica',
            'estacao', 'dummy_periodo_pico_sazonal']
print('Features de calendario criadas:')
for c in cal_cols:
    print(f'  {c}: valores unicos = {sorted(df[c].unique())}')

Features de calendario criadas:
  dia_semana: valores unicos = [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]
  fim_de_semana: valores unicos = [np.int64(0), np.int64(1)]
  mes: valores unicos = [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
  semana_epidemiologica: valores unicos = [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int

## Features atmosféricas finais

Nesta seção são criadas as features atmosféricas selecionadas para o dataset final. Em geral, elas são construídas como médias móveis defasadas, para representar exposição acumulada e evitar uso direto de informação futura.

A exceção documentada no notebook é `no2_lag0`, mantida conforme a definição adotada no processo anterior.


In [29]:
df['o3_ma_120d_shift_1d'] = df['o3'].shift(1).rolling(120).mean()

# Monoxido de nitrogenio (NO) — media movel de 30 dias, shift de 1 dia
# Um dos sinais mais fortes entre todos os poluentes
df['no_ma_30d_shift_1d'] = df['no'].shift(1).rolling(30).mean()

# Monoxido de carbono (CO) — media movel de 120 dias, shift de 1 dia
# Sinal forte e consistente em acumulacao de longo prazo
df['co_ma_120d_shift_1d'] = df['co'].shift(1).rolling(120).mean()

# Dioxido de enxofre (SO2) — media movel de 150 dias, shift de 21 dias
# Sinal forte mas dependente de janela muito longa e defasagem maior
df['so2_ma_150d_shift_21d'] = df['so2'].shift(21).rolling(150).mean()

# Oxidos de nitrogenio (NOX) — media movel de 21 dias, shift de 1 dia
# Melhor sintese para NOX, com sinal forte e consistente
df['nox_ma_21d_shift_1d'] = df['nox'].shift(1).rolling(21).mean()

# Dioxido de nitrogenio (NO2) — valor no dia D0
# Premissa: medicao de NO2 disponivel em tempo real no dia da previsao
df['no2_lag0'] = df['no2'].copy()

# Material particulado fino (PM2.5) — media movel de 30 dias, shift de 1 dia
# Sinal moderado com interpretacao de acumulacao recente
df['pm2_5_ma_30d_shift_1d'] = df['pm2_5'].shift(1).rolling(30).mean()

# Temperatura — media movel de 14 dias, shift de 1 dia
# Associacao negativa com janela curta de acumulacao
df['temp_ma_14d_shift_1d'] = df['temp'].shift(1).rolling(14).mean()

# Umidade relativa — media movel de 60 dias, shift de 1 dia
# Efeito acumulado mais longo
df['ur_ma_60d_shift_1d'] = df['ur'].shift(1).rolling(60).mean()

# Resumo das features atmosfericas
atm_cols = [
    'o3_ma_120d_shift_1d', 'no_ma_30d_shift_1d', 'co_ma_120d_shift_1d',
    'so2_ma_150d_shift_21d', 'nox_ma_21d_shift_1d', 'no2_lag0',
    'pm2_5_ma_30d_shift_1d', 'temp_ma_14d_shift_1d', 'ur_ma_60d_shift_1d'
]
print('Features atmosfericas criadas:')
for c in atm_cols:
    n_nulos = df[c].isnull().sum()
    print(f'  {c}: {n_nulos} NaNs ({n_nulos / len(df) * 100:.1f}%)')

Features atmosfericas criadas:
  o3_ma_120d_shift_1d: 120 NaNs (4.7%)
  no_ma_30d_shift_1d: 30 NaNs (1.2%)
  co_ma_120d_shift_1d: 120 NaNs (4.7%)
  so2_ma_150d_shift_21d: 170 NaNs (6.6%)
  nox_ma_21d_shift_1d: 21 NaNs (0.8%)
  no2_lag0: 0 NaNs (0.0%)
  pm2_5_ma_30d_shift_1d: 30 NaNs (1.2%)
  temp_ma_14d_shift_1d: 14 NaNs (0.5%)
  ur_ma_60d_shift_1d: 60 NaNs (2.3%)


## Montagem do dataset final

Aqui são reunidos a referência temporal, a variável alvo e todas as features selecionadas. Em seguida, são removidas as linhas iniciais sem histórico suficiente para cálculo dos lags e janelas móveis.


In [30]:
feature_cols = (
    # Endogenas — lags
    [f'lag_{k}' for k in [1, 2, 3, 7, 14, 21, 28, 30, 365]]
    # Endogenas — rolling windows
    + [f'rolling_mean_{w}' for w in [7, 14, 30]]
    + [f'rolling_std_{w}' for w in [7, 30]]
    # Calendario
    + ['dia_semana', 'fim_de_semana', 'mes', 'semana_epidemiologica',
       'estacao', 'dummy_periodo_pico_sazonal']
    # Atmosfericas
    + ['o3_ma_120d_shift_1d', 'no_ma_30d_shift_1d', 'co_ma_120d_shift_1d',
       'so2_ma_150d_shift_21d', 'nox_ma_21d_shift_1d', 'no2_lag0',
       'pm2_5_ma_30d_shift_1d', 'temp_ma_14d_shift_1d', 'ur_ma_60d_shift_1d']
)

# Monta o dataframe final
df_final = df[['data', 'target'] + feature_cols].copy()

# Remove linhas com NaN (inicio da serie sem historico suficiente)
n_antes = len(df_final)
df_final = df_final.dropna().reset_index(drop=True)
n_depois = len(df_final)

print(f'Linhas antes de remover NaN: {n_antes}')
print(f'Linhas removidas (historico insuficiente): {n_antes - n_depois}')
print(f'Linhas no dataset final: {n_depois}')
print(f'Periodo final: {df_final["data"].min().date()} a {df_final["data"].max().date()}')
print(f'\nColunas ({len(df_final.columns)}):')
print(f'  Referencia temporal: data')
print(f'  Variavel alvo: target')
print(f'  Features: {len(feature_cols)}')

Linhas antes de remover NaN: 2557
Linhas removidas (historico insuficiente): 365
Linhas no dataset final: 2192
Periodo final: 2012-12-31 a 2018-12-31

Colunas (31):
  Referencia temporal: data
  Variavel alvo: target
  Features: 29


## Verificações de consistência

Antes de salvar o resultado, esta etapa faz validações importantes do dataset final, como ausência de nulos, inexistência de datas duplicadas, continuidade temporal e checagens simples de anti-leakage.


In [31]:
print('=== Verificacao de Consistencia ===\n')

# 1. Nenhum valor nulo no dataset final
n_nulos = df_final.isnull().sum().sum()
print(f'1. Valores nulos no dataset final: {n_nulos}')
assert n_nulos == 0, 'ERRO: Existem valores nulos no dataset final!'

# 2. Serie temporal sem gaps
datas_final = pd.date_range(df_final['data'].min(), df_final['data'].max(), freq='D')
gaps = datas_final.difference(df_final['data'])
print(f'2. Gaps na serie temporal: {len(gaps)}')
if len(gaps) > 0:
    print(f'   ATENCAO: {len(gaps)} dias faltantes!')

# 3. Sem datas duplicadas
n_dup = df_final['data'].duplicated().sum()
print(f'3. Datas duplicadas: {n_dup}')
assert n_dup == 0, 'ERRO: Existem datas duplicadas!'

# 4. Target com valores razoaveis (positivos)
print(f'4. Target — min: {df_final["target"].min()}, max: {df_final["target"].max()}, '
      f'media: {df_final["target"].mean():.1f}')
assert (df_final['target'] >= 0).all(), 'ERRO: Target com valores negativos!'

# 5. Anti-leakage: lag_1 deve ser altamente correlacionado com o target,
#    mas nao identico (o que indicaria uso do proprio valor de D0)
corr_lag1 = df_final['target'].corr(df_final['lag_1'])
fracao_iguais = (df_final['target'] == df_final['lag_1']).mean()
print(f'5. Anti-leakage — corr(target, lag_1): {corr_lag1:.4f} (esperado: alto, < 1)')
print(f'   Fracao target == lag_1: {fracao_iguais:.4f} (esperado: baixo)')

# 6. Rolling mean nao contaminado com D0
corr_rm7 = df_final['target'].corr(df_final['rolling_mean_7'])
print(f'6. Anti-leakage — corr(target, rolling_mean_7): {corr_rm7:.4f} (esperado: moderada)')

# 7. Resumo descritivo de todas as colunas
print(f'\n=== Resumo Descritivo ===')
display(df_final.describe().round(2))

print('\nDataset final pronto para modelagem.')

=== Verificacao de Consistencia ===

1. Valores nulos no dataset final: 0
2. Gaps na serie temporal: 0
3. Datas duplicadas: 0
4. Target — min: 48, max: 1079, media: 339.8
5. Anti-leakage — corr(target, lag_1): 0.7115 (esperado: alto, < 1)
   Fracao target == lag_1: 0.0265 (esperado: baixo)
6. Anti-leakage — corr(target, rolling_mean_7): 0.8027 (esperado: moderada)

=== Resumo Descritivo ===


,data,target,lag_1,lag_2,lag_3,lag_7,lag_14,lag_21,lag_28,lag_30,...,dummy_periodo_pico_sazonal,o3_ma_120d_shift_1d,no_ma_30d_shift_1d,co_ma_120d_shift_1d,so2_ma_150d_shift_21d,nox_ma_21d_shift_1d,no2_lag0,pm2_5_ma_30d_shift_1d,temp_ma_14d_shift_1d,ur_ma_60d_shift_1d
count,2192,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,...,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00,2192.00
mean,2015-12-31 12:00:00,339.82,339.85,339.99,340.19,340.54,341.07,341.58,342.34,342.50,...,0.33,29.93,15.61,0.34,4.26,49.91,34.25,17.49,26.02,69.48
min,2012-12-31 00:00:00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,48.00,...,0.00,19.75,7.68,0.25,2.90,29.40,6.95,7.25,19.94,57.35
25%,2014-07-01 18:00:00,220.00,220.00,220.00,220.00,220.00,220.75,221.00,224.00,224.00,...,0.00,26.01,10.91,0.31,3.71,40.35,26.32,13.69,23.90,66.46
50%,2015-12-31 12:00:00,315.00,315.00,315.00,315.00,317.00,319.00,319.00,320.00,320.00,...,0.00,30.84,13.18,0.34,4.26,45.65,32.79,16.55,25.83,70.11
75%,2017-07-01 06:00:00,429.00,429.00,429.00,429.00,429.00,429.00,429.00,429.00,429.00,...,1.00,33.43,19.08,0.37,4.70,56.99,40.13,20.43,27.91,72.70
max,2018-12-31 00:00:00,1079.00,1079.00,1079.00,1079.00,1079.00,1079.00,1079.00,1079.00,1079.00,...,1.00,38.28,35.01,0.44,5.68,95.57,88.47,39.14,32.29,77.62
std,NaN,159.18,159.15,159.07,159.03,159.00,158.71,158.51,158.34,158.28,...,0.47,4.71,6.23,0.04,0.67,13.02,11.33,5.44,2.64,4.21



Dataset final pronto para modelagem.


## Salvamento do dataset

Por fim, o dataframe final é salvo em formato `parquet` na camada `GoldData`, ficando pronto para uso na etapa de modelagem.


In [32]:
PATH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Salva em formato parquet (eficiente e preserva tipos)
df_final.to_parquet(PATH_OUTPUT, index=False)

file_size_kb = PATH_OUTPUT.stat().st_size / 1024
print(f'Dataset salvo em: {PATH_OUTPUT}')
print(f'  Shape: {df_final.shape}')
print(f'  Tamanho: {file_size_kb:.1f} KB')

Dataset salvo em: ..\..\Data\GoldData\modelDataset.parquet
  Shape: (2192, 31)
  Tamanho: 324.4 KB
